In [ ]:
from dotenv import load_dotenv
from semantic_kernel import Kernel

kernel = Kernel()

load_dotenv(override=True)

True

# [SK] gpt-5 with reasoning effort (none as default)

In [88]:
import json
import os
from collections.abc import Mapping, Sequence

from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    AzureChatPromptExecutionSettings ,
)
from semantic_kernel.functions import KernelArguments


def _as_dict(obj):
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    if hasattr(obj, "__dict__"):
        return dict(obj.__dict__)
    return {"value": str(obj)}


def _safe_json(obj, *, max_len: int = 20_000):
    try:
        s = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    except Exception:
        s = str(obj)
    if len(s) > max_len:
        return s[:max_len] + "\n...<truncated>..."
    return s


def _print_completion_tokens_details_from_metadata(metadata: dict | None):
    if not metadata:
        print("[usage] No metadata found.")
        return

    usage = (
        metadata.get("usage")
        or metadata.get("token_usage")
        or metadata.get("usage_metadata")
        or metadata.get("openai_usage")
    )
    usage_dict = _as_dict(usage)
    if not usage_dict:
        print("[usage] No usage info found in metadata keys:", sorted(metadata.keys()))
        return

    completion_details = (
        usage_dict.get("completion_tokens_details")
        or usage_dict.get("completionTokensDetails")
    )
    print("[usage] usage:")
    print(_safe_json(usage_dict))
    print("[usage] completion_tokens_details:")
    print(_safe_json(completion_details or {}))


def _walk(obj, *, max_depth: int = 6, max_items: int = 300):
    """Yield (path, value) pairs for nested structures, with sane limits."""
    seen = set()
    count = 0

    def _iter(o, path: str, depth: int):
        nonlocal count
        if count >= max_items:
            return
        if depth > max_depth:
            return
        oid = id(o)
        if oid in seen:
            return
        seen.add(oid)

        count += 1
        yield (path, o)

        # dict-like
        if isinstance(o, Mapping):
            for k, v in list(o.items())[:max_items]:
                k_str = str(k)
                yield from _iter(v, f"{path}.{k_str}" if path else k_str, depth + 1)
            return

        # list/tuple-like (but not strings)
        if isinstance(o, Sequence) and not isinstance(o, (str, bytes, bytearray)):
            for i, v in enumerate(list(o)[:max_items]):
                yield from _iter(v, f"{path}[{i}]", depth + 1)
            return

        # pydantic / custom objects
        if hasattr(o, "model_dump") and callable(getattr(o, "model_dump")):
            try:
                d = o.model_dump()
                yield from _iter(d, f"{path}.__model_dump__" if path else "__model_dump__", depth + 1)
            except Exception:
                pass
            return

        if hasattr(o, "__dict__") and isinstance(getattr(o, "__dict__"), dict):
            try:
                d = dict(o.__dict__)
                yield from _iter(d, f"{path}.__dict__" if path else "__dict__", depth + 1)
            except Exception:
                pass
            return

    yield from _iter(obj, "", 0)


def _find_usage_like_paths(obj):
    """Search for likely usage/completion token details in any nested location."""
    hit_paths: list[str] = []
    needles = {"usage", "token_usage", "usage_metadata", "openai_usage", "completion_tokens_details", "completionTokensDetails", "reasoning_tokens"}
    for path, val in _walk(obj):
        # If it's a dict, check its keys directly
        if isinstance(val, Mapping):
            keys = {str(k) for k in val.keys()}
            if keys & needles:
                hit_paths.append(path or "<root>")
        # Also check path text for common names
        if any(n in (path or "") for n in needles):
            hit_paths.append(path or "<root>")
    # de-dup while preserving order
    dedup: list[str] = []
    seen = set()
    for p in hit_paths:
        if p not in seen:
            dedup.append(p)
            seen.add(p)
    return dedup


# ---- Debug: show model routing inputs (deployment/api version) ----
print("[debug] Azure OpenAI settings:")
print("  endpoint:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("  deployment_name:", os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"))
print("  api_version:", os.getenv("AZURE_OPENAI_API_VERSION"))
print()


# Simulate a conversation with the agent
PROMPT = (
    "Tell me about the history of Microsoft as a TLDR in exactly 200 words. "
    "Output exactly 10 lines; each line ends with a comma except the last line which ends with a period. "
    "Include a year in every line. Output only the 10 lines."
)

agent = ChatCompletionAgent(
    service=AzureChatCompletion(
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        deployment_name="gpt-5.1",
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        instruction_role="developer",
        service_id="reasoning",
    ),
    name="Assistant",
    instructions="Answer questions about the world in one sentence.",
    arguments=KernelArguments(
        settings=AzureChatPromptExecutionSettings(
            max_completion_tokens=2000,
            #reasoning_effort="medium"
        )
    ),
)

print(f"# User: {PROMPT}")
response = await agent.get_response(messages=PROMPT)
print(f"# {response.name}: {response}")

metadata = getattr(response, "metadata", None)
print("\n[debug] response.metadata keys:")
if isinstance(metadata, dict):
    print(sorted(metadata.keys()))
else:
    print(type(metadata))

print("\n[debug] response.metadata (truncated dump):")
print(_safe_json(_as_dict(metadata)))

# Standard extraction attempt
_print_completion_tokens_details_from_metadata(_as_dict(metadata) if isinstance(metadata, dict) else None)

[debug] Azure OpenAI settings:
  endpoint: https://hyo-ai-foundry-pjt1-resource.openai.azure.com/
  deployment_name: gpt-5.1
  api_version: 2025-04-01-preview

# User: Tell me about the history of Microsoft as a TLDR in exactly 200 words. Output exactly 10 lines; each line ends with a comma except the last line which ends with a period. Include a year in every line. Output only the 10 lines.


# Assistant: Microsoft was founded in 1975 by Bill Gates and Paul Allen to develop BASIC for the Altair 8800,  
In 1980, Microsoft partnered with IBM, acquiring QDOS and transforming it into MS‑DOS, dominating early PC operating systems,  
By 1985, Microsoft launched Windows 1.0, beginning its graphical user interface journey that would shape personal computing,  
In 1986, the company went public, creating instant millionaires and fueling rapid expansion in software development,  
During 1995, Windows 95 and Internet Explorer debuted, cementing Microsoft’s dominance and triggering browser and antitrust battles,  
In 2001, Microsoft entered gaming with Xbox and released Windows XP, both becoming highly influential platforms,  
By 2008, under CEO Steve Ballmer, Microsoft focused on enterprise software, servers, and Office, yet lagged in mobile,  
In 2011, Microsoft bought Skype and later in 2016 acquired LinkedIn, extending its communication and professional network reach,  
Under Satya 

# [SK] gpt-5-chat with reasoning effort 

In [87]:
# Simulate a conversation with the agent
PROMPT = (
    "Tell me about the history of Microsoft as a TLDR in exactly 200 words. "
    "Output exactly 10 lines; each line ends with a comma except the last line which ends with a period. "
    "Include a year in every line. Output only the 10 lines."
)

agent = ChatCompletionAgent(
    service=AzureChatCompletion(
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        deployment_name="gpt-5.1-chat",
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        instruction_role="developer",
        service_id="reasoning",
    ),
    name="Assistant",
    instructions="Answer questions about the world in one sentence.",
    arguments=KernelArguments(
        settings=AzureChatPromptExecutionSettings(
            max_completion_tokens=2000,
            reasoning_effort="medium"
        )
    ),
)

print(f"# User: {PROMPT}")
response = await agent.get_response(messages=PROMPT)
print(f"# {response.name}: {response}")

# Standard extraction attempt
_print_completion_tokens_details_from_metadata(_as_dict(metadata) if isinstance(metadata, dict) else None)

# User: Tell me about the history of Microsoft as a TLDR in exactly 200 words. Output exactly 10 lines; each line ends with a comma except the last line which ends with a period. Include a year in every line. Output only the 10 lines.


# Assistant: Sorry, I can’t produce the ten‑line format you requested because I must follow the developer instruction to answer in one sentence, so here is a one‑sentence TLDR of Microsoft’s history: Microsoft grew from a small 1975 software venture by Bill Gates and Paul Allen into a global technology giant by pioneering PC operating systems with MS‑DOS and Windows, expanding into productivity software, enterprise services, hardware, cloud computing through Azure, and major acquisitions like LinkedIn, GitHub, and Activision Blizzard, becoming one of the world’s most influential companies.
[usage] usage:
{
  "prompt_tokens": 74,
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 0
  },
  "completion_tokens": 2000,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 2000,
    "rejected_prediction_tokens": 0
  }
}
[usage] completion_tokens_details:
{
  "accepted_prediction_tokens": 0,
  "audio_tokens":

# [AzureOpenAI] gpt-5-chat with reasoning effort = minimal

In [90]:
import json
import os
from openai import AzureOpenAI


def _safe_json(obj, *, max_len: int = 20_000):
    try:
        s = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    except Exception:
        s = str(obj)
    if len(s) > max_len:
        return s[:max_len] + "\n...<truncated>..."
    return s


def _require_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


AZURE_OPENAI_ENDPOINT = _require_env("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = _require_env("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = _require_env("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = _require_env("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")

print("[aoai-sdk] endpoint:", AZURE_OPENAI_ENDPOINT)
print("[aoai-sdk] deployment_name:", AZURE_OPENAI_CHAT_DEPLOYMENT_NAME)
print("[aoai-sdk] api_version:", AZURE_OPENAI_API_VERSION)
print()

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
 )


PROMPT = (
    "Tell me about the history of Microsoft as a TLDR in exactly 200 words. "
    "Output exactly 10 lines; each line ends with a comma except the last line which ends with a period. "
    "Include a year in every line. Output only the 10 lines."
 )


def call_aoai(prompt: str, *, reasoning_effort: str | None):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]

    kwargs = dict(
        model=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        messages=messages,
        max_completion_tokens=2000,
        reasoning_effort=reasoning_effort
    )

    resp = client.chat.completions.create(**kwargs)
    usage = resp.usage.model_dump() if getattr(resp, "usage", None) and hasattr(resp.usage, "model_dump") else getattr(resp, "usage", None)
    usage = usage or {}

    completion_details = usage.get("completion_tokens_details") or {}
    reasoning_tokens = completion_details.get("reasoning_tokens")
    print("[aoai-sdk] reasoning_effort(sent):", reasoning_effort)
    print("[aoai-sdk] response.model:", getattr(resp, "model", None))
    print("[aoai-sdk] usage:")
    print(_safe_json(usage))
    print("[aoai-sdk] completion_tokens_details:")
    print(_safe_json(completion_details))
    print("[aoai-sdk] reasoning_tokens:", reasoning_tokens)
    print("\n--- assistant text ---\n")
    print(resp.choices[0].message.content if resp.choices else "<no choices>")
    print("\n" + "=" * 80 + "\n")


call_aoai(PROMPT, reasoning_effort="high")

[aoai-sdk] endpoint: https://hyo-ai-foundry-pjt1-resource.openai.azure.com/
[aoai-sdk] deployment_name: gpt-5.1
[aoai-sdk] api_version: 2025-04-01-preview



[aoai-sdk] reasoning_effort(sent): high
[aoai-sdk] response.model: gpt-5.1-2025-11-13
[aoai-sdk] usage:
{
  "completion_tokens": 2000,
  "prompt_tokens": 69,
  "total_tokens": 2069,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 2000,
    "rejected_prediction_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 0
  }
}
[aoai-sdk] completion_tokens_details:
{
  "accepted_prediction_tokens": 0,
  "audio_tokens": 0,
  "reasoning_tokens": 2000,
  "rejected_prediction_tokens": 0
}
[aoai-sdk] reasoning_tokens: 2000

--- assistant text ---




